# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
# Same feature engineering as the previous lab: drop nulls, reduce Cabin to
# its deck letter, drop the two non-predictive id/text columns, dummy-encode
# the rest.
spaceship = spaceship.dropna()
spaceship["Cabin"] = spaceship["Cabin"].str[0]
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

categorical_cols = spaceship.select_dtypes(include=["object", "bool"]).columns.drop("Transported")
spaceship_encoded = pd.get_dummies(spaceship, columns=categorical_cols, drop_first=True)

features = spaceship_encoded.drop(columns=["Transported"])
target = spaceship_encoded["Transported"]

print(features.shape)
features.head()

(6606, 19)


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,False,False,False,False,False,False,False,True,False,False,False,True,False


In [4]:
# Feature Scaling -- this is the piece the previous lab's KNN model was
# missing. Fit the scaler on train only, to avoid leaking test-set
# information into the transform.
from sklearn.preprocessing import StandardScaler

**Perform Train Test Split**

In [5]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train.shape, X_test.shape

((5284, 19), (1322, 19))

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# bootstrap=True (the default) is Bagging -- sampling WITH replacement.
# Pasting is the same class with bootstrap=False (sampling without
# replacement); Bagging is the more common default choice.
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=0)
bagging.fit(X_train, y_train)

print(f"Bagging train accuracy: {bagging.score(X_train, y_train):.4f}")
print(f"Bagging test accuracy : {bagging.score(X_test, y_test):.4f}")

Bagging train accuracy: 0.9404
Bagging test accuracy : 0.7890


- Random Forests

In [7]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest is Bagging plus one more trick: each split only considers
# a random subset of features, not all of them -- this decorrelates the
# trees further (two trees can't both just latch onto the single
# strongest feature every time).
rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_train, y_train)

print(f"Random Forest train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"Random Forest test accuracy : {rf.score(X_test, y_test):.4f}")

Random Forest train accuracy: 0.9404
Random Forest test accuracy : 0.7927


- Gradient Boosting

In [8]:
from sklearn.ensemble import GradientBoostingClassifier

# Bagging/RF train trees independently and average them; boosting trains
# trees sequentially, each one focused on correcting the previous
# ensemble's errors (via the gradient of the loss). Different strategy
# entirely -- reducing bias by chaining weak learners, not reducing
# variance by averaging independent ones.
gb = GradientBoostingClassifier(random_state=0)
gb.fit(X_train, y_train)

print(f"Gradient Boosting train accuracy: {gb.score(X_train, y_train):.4f}")
print(f"Gradient Boosting test accuracy : {gb.score(X_test, y_test):.4f}")

Gradient Boosting train accuracy: 0.8236
Gradient Boosting test accuracy : 0.7867


- Adaptive Boosting

In [9]:
from sklearn.ensemble import AdaBoostClassifier

# AdaBoost is also sequential boosting, but instead of fitting to the
# residual gradient like Gradient Boosting, it reweights the training
# samples after each round -- misclassified points get more weight, so
# the next weak learner focuses harder on exactly what the ensemble is
# still getting wrong.
ada = AdaBoostClassifier(random_state=0)
ada.fit(X_train, y_train)

print(f"AdaBoost train accuracy: {ada.score(X_train, y_train):.4f}")
print(f"AdaBoost test accuracy : {ada.score(X_test, y_test):.4f}")

AdaBoost train accuracy: 0.7886
AdaBoost test accuracy : 0.7776


Which model is the best and why?

In [10]:
#comment here
results = {
    "Bagging": bagging.score(X_test, y_test),
    "Random Forest": rf.score(X_test, y_test),
    "Gradient Boosting": gb.score(X_test, y_test),
    "AdaBoost": ada.score(X_test, y_test),
}
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{name:20s} {acc:.4f}")

Random Forest        0.7927
Bagging              0.7890
Gradient Boosting    0.7867
AdaBoost             0.7776


**Random Forest wins** on the test set (79.3%), just ahead of plain Bagging (78.9%), with Gradient Boosting close behind (78.7%) and AdaBoost trailing (77.8%). All four beat the single-tree/KNN baselines from the previous two labs (KNN test accuracy was 76.6% with numeric-only features, 76.9% after feature engineering) — this is the expected story for ensembles: combining many weak/high-variance learners into one vote reduces variance and generally beats any single model built the same way.

Random Forest edges out plain Bagging specifically because of the extra per-split feature subsampling — it decorrelates the individual trees a bit more than Bagging alone, so averaging them cancels out more noise. AdaBoost coming in last here isn't a universal result (it depends heavily on the dataset and default hyperparameters, especially the weak learner's depth) — it's just what this particular scaled, dummy-encoded Spaceship Titanic setup happened to produce.